# Is CO2 rising faster than it used to?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F09_rising_faster.ipynb).

In 1958 Charles David Keeling put a carbon-dioxide analyser high on a
Hawaiian volcano, far from anybody's chimney, and started measuring. It has been running ever
since, and the monthly numbers it has produced — one more of them every month, including the
month you are reading this in — are among the most consequential in science.

It goes up. Everybody knows it goes up. The question worth asking is whether it goes up in a
*straight line* — because a straight line and a gently bending curve look nearly identical over
the years you have measured, and say completely different things about the years you have not.
The difference between them is the difference between reaching 500 parts per million in
the 2050s and reaching it in the next century.

Today you decide how much bend this record can actually support. Not by arguing about it: by
hiding 25 years of it from yourself, fitting curves to what is left, and checking
what they say about the years you hid.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say how fast CO2 is rising, whether that rate is itself rising, and what the
record does and does not let you say about when it reaches 500 ppm.

**The skills.** Fit a curve instead of a line with `np.polyfit` and read it back with
`np.polyval`; measure how badly a model misses in the units of the data; split a record into
years you fit on and years you keep back; and read the two error curves that tell you when a
model has stopped learning the pattern and started memorising the data.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load():
    """Read NOAA's Mauna Loa monthly CO2 file live; fall back to the copy stored with the course, downloaded 2026-08-31."""
    try:
        return pd.read_csv("https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv", comment="#")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + "week09_co2_mm_mlo.csv")

# NOAA Global Monitoring Laboratory, Mauna Loa monthly mean CO2 — the Keeling curve.
# The comment="#" above skips the file's own header notes. NOAA adds a month to this file
# every month, so the record below is longer every time you run it.
monthly = load()[["year", "month", "average"]]

# The record starts partway through its first year and its last year is still being measured,
# so averaging those two would average whichever seasons happen to be there. Keep the rest.
first_full = monthly["year"].min() + 1
last_full = monthly["year"].max() - 1
annual = (monthly[(monthly["year"] >= first_full) & (monthly["year"] <= last_full)]
          .groupby("year", as_index=False)["average"].mean()
          .rename(columns={"average": "co2"}))

START_YEAR = first_full   # every fit below counts years from here, not from year zero

print("monthly readings:", len(monthly))
print("complete years:", len(annual), "—", first_full, "to", last_full)
print(annual.head())

## The record

Three lines of pandas turned every monthly reading into one number per complete year. Draw both
and you can see why we bothered — and why the setup cell threw two years away.

In [ ]:
plt.plot(monthly["year"] + monthly["month"] / 12, monthly["average"],
         color="0.7", lw=0.8, label="every month")
plt.plot(annual["year"], annual["co2"], marker="o", ms=3, label="annual average")
plt.xlabel("year")
plt.ylabel("CO2 (parts per million)")
plt.title(f"Mauna Loa CO2, {len(monthly)} monthly readings, {len(annual)} complete years")
plt.legend()
plt.show()

The grey line has teeth. That is the northern hemisphere breathing: most of the world's land
plants live north of the equator, they pull CO2 out of the air through the growing season and
give it back as leaves and soil decay through the winter, and Mauna Loa sits far enough north to
feel it. Measure one whole year of it, then average only as many months as the record's
unfinished last year has, and compare the two.

In [ ]:
recent = monthly[monthly["year"] == last_full]      # the last whole year
unfinished = monthly[monthly["year"] > last_full]   # the year still being measured
months_so_far = unfinished["month"].max()

print(last_full, "ran from", round(recent["average"].min(), 2), "to",
      round(recent["average"].max(), 2), "ppm — a swing of",
      round(recent["average"].max() - recent["average"].min(), 2), "ppm inside one year")
print("its average over all twelve months:", round(recent["average"].mean(), 2), "ppm")
print("its average over the first", months_so_far, "months only:",
      round(recent[recent["month"] <= months_so_far]["average"].mean(), 2), "ppm")

6.14 ppm of swing inside a single year — and averaging only the months a part-year
happens to have shifts that year's mean by around a ppm, because the months you keep are not a
fair sample of the seasons. That is why the setup cell kept only whole years: the record begins
partway through 1958 and its last year is still being measured, so both would
have carried that bias into every curve we fit. It also means nothing below changes when NOAA
publishes next month's number.

The teeth otherwise are not what this week is about. Averaging each year removes them and leaves
the rise, which is what we want to model. So from here on we work with `annual`: one number per
year, from 1959 to 2025.

You fitted a straight line in the regression week; here it is again, with a new pair of functions.
`np.polyfit(x, y, degree)` finds the best polynomial of that degree — at degree 1 it *is* the
least-squares line — and hands back its coefficients. `np.polyval(coeffs, x)` reads the fitted
curve back at any x you like, including x values that were never in the data. Wrapping both in
functions of our own means the rest of the notebook is one line per fit.

In [ ]:
def fit_curve(table, degree):
    """Fit a polynomial of this degree to a table of years and CO2; hand back its coefficients."""
    return np.polyfit(table["year"] - START_YEAR, table["co2"], degree)


def curve_value(coeffs, year):
    """What a fitted curve says CO2 is in that year — one year, or a whole column of them."""
    return np.polyval(coeffs, year - START_YEAR)

### ✏️ Your turn 1

Fit a straight line to the whole of `annual` and read two numbers off it: how fast it says CO2
rises, in ppm per year, and what it says CO2 was at the start of the record.

`fit_curve(annual, 1)` gives you the coefficients. For a straight line the first coefficient is
the slope, so `line[0]` is the ppm per year. For the second number, ask `curve_value` what the
line says at `START_YEAR`.

**Use these names**, because the self-check looks for them: `line` and `slope`.

In [ ]:
# ← your answer here


assert 0.5 < slope < 5, \
    "CO2 rises by a couple of ppm a year, so a slope in the hundreds is the wrong coefficient: \
line[0] is the ppm per year and line[1] is where the line starts"
print("✓ the straight line —", round(slope, 3), "ppm per year, starting from",
      round(curve_value(line, START_YEAR), 1), "ppm")

So a single straight line says CO2 has climbed 1.672 ppm every year since
1959, starting from 306.1 ppm. Both numbers look reasonable. The
question is whether the line is *right*, and for that we need a number that says how badly a
model misses.

The one we will use all week is the **typical miss**: square every gap between what happened and
what the curve said, average the squares, take the square root. Squaring stops a miss of +5 and a
miss of −5 cancelling, and the square root at the end puts the answer back in ppm, so it means
something you can read.

In [ ]:
def typical_miss(actual, predicted):
    """The usual size of the gap between what happened and what the curve said, in ppm."""
    return np.sqrt(np.mean((actual - predicted) ** 2))


print("the straight line misses by", round(typical_miss(annual["co2"],
                                                        curve_value(line, annual["year"])), 2),
      "ppm on average")

4.55 ppm, on a record that spans more than a hundred. By the standards of the
regression week that is a good fit. But an average miss hides *where* the misses are, and that is the whole
question here. Subtract the line from the data and plot what is left over.

In [ ]:
residual = annual["co2"] - curve_value(line, annual["year"])

for start in [first_full, 1980, last_full - 9]:
    decade = residual[(annual["year"] >= start) & (annual["year"] < start + 10)]
    print(start, "to", start + 9, ": the line misses by", round(decade.mean(), 1), "ppm on average")

print("furthest above the line:", round(residual.max(), 1), "ppm")
print("furthest below the line:", round(residual.min(), 1), "ppm")

In [ ]:
plt.plot(annual["year"], residual, marker="o", ms=3)
plt.axhline(0, color="0.5", lw=0.8)      # the line itself, for the eye to measure against
plt.xlabel("year")
plt.ylabel("data minus straight line (ppm)")
plt.title(f"What the straight line leaves behind, {len(annual)} years")
plt.show()

Those are not random misses. They are a smile: the line runs below the data at both ends and
above it in the middle: +5.8 ppm on average over the record's first
ten years, -3.1 ppm over the 1980s, +6.5 ppm
over the last ten. The two extremes are +10.9 and -5.7 ppm.
A least-squares line always leaves misses that average to zero overall, so seeing them arranged
in an arc rather than scattered means the line has the wrong *shape*, not just some noise around
it.

A straight line has one rate of rise and cannot change it. This record changes it.

## Letting the curve bend

The **degree** of a polynomial is how many bends it is allowed. Degree 1 is a straight line, no
bends. Degree 2 is a parabola: one bend, one steady change of slope. Degree 3 can bend twice,
degree 9 can bend eight times. `fit_curve` already takes the degree as its second argument, so
trying more flexible curves costs nothing but the number.

In [ ]:
for degree in [1, 2, 3]:
    coeffs = fit_curve(annual, degree)
    plt.plot(annual["year"], curve_value(coeffs, annual["year"]), label="degree " + str(degree))

plt.plot(annual["year"], annual["co2"], "k.", ms=4, label="annual average")
plt.xlabel("year")
plt.ylabel("CO2 (parts per million)")
plt.title(f"Three curves fitted to the same {len(annual)} years")
plt.legend()
plt.show()

The parabola and the cubic go through the data; the straight line does not. Look for the orange
line and you will not find it — over these years the cubic sits on top of the parabola almost
exactly, which is worth remembering, because in a moment we will have to tell those two apart.
Put numbers on the three of them and the improvement is not subtle.

In [ ]:
for degree in [1, 2, 3, 5, 9]:
    coeffs = fit_curve(annual, degree)
    print("degree", degree, "misses by",
          round(typical_miss(annual["co2"], curve_value(coeffs, annual["year"])), 3), "ppm")

Degree 2 cuts the miss from 4.553 ppm to 0.743, and it keeps falling
after that: 0.684 at degree 3, 0.429 at degree 9. That is not a
coincidence and it is not evidence. **A more flexible curve can always sit closer to the data it
was fitted on** — a degree-9 polynomial has ten coefficients to adjust where a line has two, so
of course it gets nearer. Fit quality on the data you fitted cannot tell you which degree to use,
because it will
always vote for the most flexible one available.

What the bend itself means, though, is real, and the data says so without any model at all. Fit a
separate straight line to each decade and read off its slope.

In [ ]:
for start in [1960, 1970, 1980, 1990, 2000, 2010]:
    decade = annual[(annual["year"] >= start) & (annual["year"] < start + 10)]
    print(str(start) + "s: CO2 rose", round(fit_curve(decade, 1)[0], 2), "ppm per year")

0.81 ppm a year in the 1960s, 2.43 in the 2010s — the rise
itself has roughly tripled, with the 1990s the one decade that barely moved on the one before it.
That is the acceleration itself, straight from the data with no model in it, and it has a
straightforward cause. Part of each year's fossil-fuel and land-use emissions is taken up by the
ocean and by the land biosphere; the rest stays in the air. So the concentration follows the
emissions that have accumulated, and emissions have grown decade on decade. A curve with a bend
in it is not a statistical convenience here; it is what an accelerating source looks like.

Which leaves the harder question. The rise is bending, so degree 1 is too simple. How much bend
should we allow — and how would we know?

## Keeping some years back

The trouble with judging a curve by how close it sits to the data is that the data has already
been used. The fix is the oldest trick in the subject and it is one sentence: **Hide some data
from yourself, then check.**

Fit only the years up to 2000, then ask each fitted curve what CO2 was in `last_full` —
a year it has never seen — and compare with what actually happened. 25 years of the
future, already in hand, waiting to mark the answer.

In [ ]:
train = annual[annual["year"] <= 2000]
later = annual[annual["year"] > 2000]
observed = annual["co2"].iloc[-1]

print("fitting on", len(train), "years, holding back", len(later))
print("what actually happened in", last_full, "was", round(observed, 2), "ppm")

### Predict before you run

Three curves are about to be fitted to 1959–2000 and asked about the last
complete year: a straight line, a parabola and a cubic. One of them will come closest. Which?
Change `my_guess` to 1, 2 or 3 and run the cell — committing to a wrong answer is worth more
than being told the right one.

In [ ]:
my_guess = 3

print("I think degree", my_guess, "will come closest")

### ✏️ Your turn 2

Fit degrees 1, 2 and 3 to `train` only — nothing after 2000 — and ask each one what CO2
was in `last_full`, the last complete year of the record. Print, for each degree, what it
predicted and how far that is from `observed`.

Loop over `[1, 2, 3]`, and inside the loop use `fit_curve(train, degree)` and then
`curve_value(coeffs, last_full)`. Collect the three predictions in a list as you go.

**Use these names**, because the self-check looks for them: `predictions`.

In [ ]:
# ← your answer here


assert max(predictions) - min(predictions) > 20, \
    "three curves that never saw a year after the split disagree about it by tens of ppm; if \
yours nearly agree they were fitted on the whole record and already know the answer — check you \
passed train, not annual, to fit_curve"
print("✓ three forecasts — the closest misses by",
      round(min(abs(p - observed) for p in predictions), 1), "ppm, the worst by",
      round(max(abs(p - observed) for p in predictions), 1), "ppm")

Read those three numbers again, because they are the point of the week.

The straight line predicted 400.8 ppm and was 26.6 ppm low: too simple, it
never saw the acceleration coming. The parabola predicted 423.6 and was
3.8 ppm low — off by less than the seasonal swing in a single year, from a fit
that stopped 25 years before the answer.

And the cubic — the most flexible of the three, the one that sat closest to the training years —
predicted 392.9, missing by 34.5 ppm. **Worse than the straight line.**
*A curve that memorises the data you gave it fails on the data you did not.* The picture says it
better than the numbers do.

In [ ]:
span = np.arange(first_full, last_full + 1)
for degree in [1, 2, 3]:
    coeffs = fit_curve(train, degree)
    plt.plot(span, curve_value(coeffs, span), label="degree " + str(degree))

plt.plot(train["year"], train["co2"], "k.", ms=4, label="fitted on these")
plt.plot(later["year"], later["co2"], "r.", ms=5, label="held back")
plt.axvline(2000, color="0.5", lw=0.8)     # where the curves stopped seeing data
plt.xlabel("year")
plt.ylabel("CO2 (parts per million)")
plt.title(f"Fitted on {len(train)} years, checked against {len(later)}")
plt.legend()
plt.show()

Three curves that were nearly on top of each other over the years they were fitted on fan apart
the moment they leave them. *Too simple and you miss the pattern; too flexible and you memorise
the noise.* The held-out years are the only thing on that plot that can tell the three apart.

## Choosing how much bend

One year of checking is thin. Score each curve on all 25 held-out years instead, and
score it on its training years too, so the two can be compared.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
def fit_curve(table, degree):
    """Fit a polynomial of this degree to a table of years and CO2; hand back its coefficients."""
    return np.polyfit(table["year"] - START_YEAR, table["co2"], degree)


def curve_value(coeffs, year):
    """What a fitted curve says CO2 is in that year — one year, or a whole column of them."""
    return np.polyval(coeffs, year - START_YEAR)


def typical_miss(actual, predicted):
    """The usual size of the gap between what happened and what the curve said, in ppm."""
    return np.sqrt(np.mean((actual - predicted) ** 2))


train = annual[annual["year"] <= 2000]
later = annual[annual["year"] > 2000]

### ✏️ Your turn 3

Loop over `degrees` and, for each one, fit on `train` and record two typical misses: one against
`train` itself, one against `later`. Print both as you go, then plot the two lists against
`degrees`.

The held-out misses run from about one ppm to tens of thousands, so put the y-axis on a log
scale with `plt.yscale("log")` — the log axes from the plotting week, for exactly the reason
you met them there.

**Use these names**, because the self-check looks for them: `degrees`, `train_miss`, `test_miss`.

```
degrees = [1, 2, 3, 4, 5, 6, 7, 8, 9]
train_miss = []
test_miss = []
```

In [ ]:
# ← your answer here


assert train_miss.index(min(train_miss)) != test_miss.index(min(test_miss)), \
    "the best degree on the training years should not also be the best on the held-out years; \
if they match, both lists were scored against the same rows"
print("✓ two error curves — training is best at degree",
      degrees[train_miss.index(min(train_miss))], "and the held-out years at degree",
      degrees[test_miss.index(min(test_miss))])

*Watch two lines. When training keeps falling and test turns up, stop.* The training line slides
downhill the whole way, from 1.719 ppm at degree 1 to 0.314 at
degree 9, exactly as it must. The held-out line dives to
1.57 ppm at degree 2 and then climbs: 16.93 at
degree 3, 107 at degree 5, 13,682 at degree 9. It does not
climb smoothly — degree 4 dips back below degree 3 — but by degree 5 the direction is not in
doubt. The widening gap between the two lines is the model learning the training years by heart.

One cut is one cut, though. Before believing degree 2, move the split and
see whether the answer moves with it.

In [ ]:
for cut in [1990, 1995, 2005]:
    tr = annual[annual["year"] <= cut]
    te = annual[annual["year"] > cut]
    misses = [typical_miss(te["co2"], curve_value(fit_curve(tr, d), te["year"])) for d in degrees]
    print("splitting at", cut, "the best degree is", degrees[misses.index(min(misses))])

### ✏️ Your turn 4

Which degree would you use, and why? Two or three sentences in the cell below, quoting your own
numbers from your turn 3 and from the four splits above.

A good answer says what the two error curves do differently, gives the held-out miss of the
degree you picked and of at least one you rejected, and says something about whether moving the
split changed your mind.

*(Double-click this cell and replace this line with your answer.)*

Between them the four cuts say something a single cut could not. Splitting at 1995
and 2005 gives degree 2, as 2000 did;
splitting at 1990 gives degree 3. So what this
record supports is "two bends, possibly three", not "exactly two" — and every one of the four
cuts rules out the flexible end completely.

## Choosing which years to hide

We held back the *last* 25 years. The obvious alternative is to hold back
25 years chosen at random from anywhere in the record — that is what most textbook
train/test splits do, and it is what you would reach for if the rows were unrelated to each
other. `.sample(frac=1, random_state=...)` shuffles a table into a random order, and
`random_state` fixes the shuffle so everyone in the room gets the same one.

In [ ]:
shuffled = annual.sample(frac=1, random_state=88)
rand_test = shuffled.iloc[:len(later)]      # the same number of years as the honest split
rand_train = shuffled.iloc[len(later):]

print("random split:", len(rand_train), "years to fit on,", len(rand_test), "held out")
print("held-out years, in order:", sorted(rand_test["year"])[:8], "...")

### ✏️ Your turn 5

Score degrees 1, 2, 3 and 9 on this random split — fit on `rand_train`, measure the typical miss
against `rand_test` — and print each one beside the held-out miss the same degree got in your
turn 3, which is `test_miss[degree - 1]`.

**Use these names**, because the self-check looks for them: `rand_train`, `rand_test`,
`rand_miss`.

In [ ]:
# ← your answer here


assert len(rand_miss) == 4, "rand_miss should hold one miss per degree — 1, 2, 3 and 9"
assert rand_miss[3] < test_miss[8] / 100, \
    "scattering the held-out years through the record should flatter degree 9 by orders of \
magnitude; if it does not, check you fitted on rand_train and scored against rand_test"
print("✓ two ways to split — at degree 9 the random split says",
      round(rand_miss[3], 3), "ppm and the held-back future says",
      round(test_miss[8], 3), "ppm")

The random split says degree 9 is the *best* of the four, missing by 0.564 ppm. The
honest split says degree 9 misses by 13,682 ppm. Same data, same curve, an
answer roughly 24,000 times apart, and only one of them is true.

The reason is that a year is not independent of its neighbours. Shuffle the record and 1987 can
end up in the held-out set while 1986 and 1988 stay in the training set — and a curve that has
been fitted through both neighbours arrives at 1987 nearly right without knowing anything about
CO2, because all it has to do is bridge a gap two years wide. The years you held back gave their
answers away by sitting next to the years you fitted on. That is called **leakage**: any route by
which information about the data you are scoring on reaches the model before you score it. It is
one of the commonest ways a model that does not work reports that it does.

The cure is not statistical, it is a question about the job. We want to know what CO2 will do
*next*, so the test has to be exactly that: fit on the past, predict the future, never the other
way round.

## The question, answered

**Yes — the rise itself is rising, by roughly a factor of three.** CO2 at Mauna Loa went up
0.81 ppm a year in the 1960s and 2.43 ppm a year in the
2010s, and a straight line through the whole record leaves an arc of misses that no amount of
noise explains. One bend is enough to capture it: a parabola fitted with nothing after
2000 predicted 2025 to within 3.8 ppm, while a straight
line was 26.6 ppm low and a cubic — more flexible, closer to the training
years, worse at the job — was 34.5 ppm out.

What that does *not* license is trusting a polynomial far beyond the record. The parabola has no
physics in it; it fits because emissions have grown smoothly so far, and it will keep bending
upward whatever happens next, because that is what parabolas do. The homework takes it out to
500 ppm anyway, and the spread you get back is the honest measure of how much of a
forecast is data and how much is the modelling choice.

## Week 9 summary

**The question.** Is CO2 rising faster than it used to?

### What to remember

| | |
|---|---|
| **1** | Too simple misses the pattern; too flexible memorises the noise. |
| **2** | Hold out data you already have, then check — it is the cheapest honest test there is. |
| **3** | The CO2 curve is bending, and which model you choose decides the decade by which it crosses 500 ppm. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Leakage** | Any route by which information about the data you are scoring on reaches the model before you score it. |
| **Polynomial degree** | The degree of a polynomial is how many bends it is allowed: degree 1 is a straight line, degree 2 bends once. |
| **Overfitting** | A curve that memorises the data you gave it fails on the data you did not. |
| **Train/test split** | Hide some data from yourself, then check. |
| **Bias-variance** | Too simple and you miss the pattern; too flexible and you memorise the noise. |
| **Learning curve** | Watch two lines. When training keeps falling and test turns up, stop. |

### Code you met this week

| Function | What it does |
|---|---|
| `table.iloc[a:b]` | rows by position, the way a list slice works |
| `np.polyfit(x, y, degree)` | fit the best polynomial of that degree; at degree 1 it is the least-squares line |
| `np.polyval(coeffs, x)` | read a fitted curve back at any x, including x values never in the data |
| `np.sqrt(x) / np.mean(x)` | the square root and the average — the two halves of a typical miss |
| `table.sample(frac=1, random_state=n)` | shuffle a table; random_state fixes the shuffle so everyone gets the same one |

## Homework

Three parts, on the same record you have had open all along. If you have restarted since class,
run the setup cell at the top and then the cell just below, which rebuilds the functions class
wrote and the split it used.

Class never once asked a curve about a year beyond the record. That is the whole homework.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
def fit_curve(table, degree):
    """Fit a polynomial of this degree to a table of years and CO2; hand back its coefficients."""
    return np.polyfit(table["year"] - START_YEAR, table["co2"], degree)


def curve_value(coeffs, year):
    """What a fitted curve says CO2 is in that year — one year, or a whole column of them."""
    return np.polyval(coeffs, year - START_YEAR)


def typical_miss(actual, predicted):
    """The usual size of the gap between what happened and what the curve said, in ppm."""
    return np.sqrt(np.mean((actual - predicted) ** 2))


train = annual[annual["year"] <= 2000]
later = annual[annual["year"] > 2000]

### ✏️ Your turn 6

500 ppm is a round number people quote as a milestone; nothing physical happens at
exactly 500. When does this record say we get there?

Write `crossing_year(coeffs)`. It should walk the years from 2027 to 2300 in a loop, ask
`curve_value(coeffs, year)` what CO2 is in each, and `return` the first year that reaches
500 or more. If none of them does, return `None`. Give it a docstring.

Then fit degrees 1, 2 and 3 to the **whole** of `annual` and print what each says.

**Use these names**, because the self-check looks for them: `crossing_year`.

In [ ]:
# ← your answer here


assert 2027 <= crossing_year(fit_curve(annual, 1)) <= 2300, \
    "crossing_year should hand back a YEAR, not a position in the loop"
print("✓ the 500 ppm crossing — a straight line through the whole record puts it in",
      crossing_year(fit_curve(annual, 1)))

### ✏️ Your turn 7

Part 1 fitted the whole record. Class validated its degrees on a fit that used nothing after
2000. Both are defensible — the whole record uses every year you have, while the
1959–2000 fit is the only one whose forecasting anybody has actually tested
— and they do not agree.

Build `crossings`, a list of four years: degree 1 and degree 2, each fitted to `annual` and to
`train`. Print all four with a label saying which is which, print the spread between the earliest and the
latest, and finally print the one year you would quote.

This part re-uses `crossing_year` from part 1; its self-check tells you whether that is working.

**Use these names**, because the self-check looks for them: `crossings`.

In [ ]:
# ← your answer here


assert max(crossings) - min(crossings) > 20, \
    "four years within twenty of each other means the degree or the training table did not change"
print("✓ four forecasts — they span", max(crossings) - min(crossings),
      "years, from", min(crossings), "to", max(crossings))

### ✏️ Your turn 8

Two or three sentences in the cell below, and every claim in them has to be one of *your* numbers.

Quote at least three: the year you chose to quote in part 2, the spread between your four
crossing years, and the held-out miss from your turn 3 of the degree you chose. Then answer this:
a newspaper wants one year. What do you give them, and what does the spread tell their reader
about how much of that year came from the record and how much from your choice of curve?

*(Double-click this cell and replace this line with your answer.)*